# Portfolio analytics

This notebook only orchestrates: it calls `transactions` (trade logic), `prices` (Yahoo Finance fetch + cache), `returns` (CAGR / HYSA benchmark math), and `visualization` (all charts). No logic lives in this notebook itself — see `docs/architecture.md` for the module map.

In [1]:
from datetime import date
from pathlib import Path

from trades import prices, returns, transactions, visualization
from trades.config import AggregationConfig, PriceApiConfig, ReturnsConfig

TRADES_CSV = Path("..") / "data" / "20260701_trades.csv"
AS_OF_DATE = date.today()  # change this to price the portfolio as of any past date

# Every tunable parameter lives on one of these config objects (see
# docs/architecture.md#configuration) — nothing here is a hidden default.
aggregation_config = AggregationConfig()
price_api_config = PriceApiConfig()
returns_config = ReturnsConfig()

## 1. Load, enrich, and aggregate trades

`load_raw_trades` validates every CSV row through the `RawTrade` model. `enrich_trades` adds `usd_per_share`. `aggregate_same_day_trades` then merges same-day, same-symbol fills executed within 0.01% of each other into one row (summed shares/USD, recomputed $/share) — this is the dataset used for everything downstream.

In [2]:
raw = transactions.load_raw_trades(TRADES_CSV)
enriched = transactions.enrich_trades(raw)
trades = transactions.aggregate_same_day_trades(enriched, aggregation_config)
print(f"{len(raw)} raw fills -> {len(trades)} aggregated trades")
trades

23 raw fills -> 16 aggregated trades


,trade_date,symbol,shares,usd_spent,usd_per_share,n_trades
0,2026-01-27,VOO,0.1500,96.06,640.400000,1
1,2026-04-10,BND,1.6950,125.00,73.746313,1
2,2026-04-10,VOO,1.3974,874.97,626.141405,1
3,2026-04-10,VXUS,3.0706,249.99,81.414056,1
4,2026-05-05,BND,0.4129,30.24,73.238072,1
5,2026-05-05,BND,3.0000,219.75,73.250000,1
6,2026-05-05,VOO,2.6281,1749.96,665.865074,3
7,2026-05-05,VXUS,5.9858,499.99,83.529353,2
8,2026-05-06,BND,0.0039,0.29,74.358974,1
9,2026-06-01,QQQM,6.1700,1889.99,306.319287,2


## 2. Investment schedule

Totals per symbol, invested-per-month (overall and per symbol), the daily investment timeline (with gaps between buys), and a pie breakdown with a menu to switch between whole-portfolio-by-symbol and any one symbol's by-date split.

In [3]:
total_by_symbol = transactions.total_invested_by_symbol(trades)
print("Total invested to date, by symbol:")
print(total_by_symbol.to_string())
print(f"\nTotal invested to date, whole portfolio: ${total_by_symbol.sum():,.2f}")

Total invested to date, by symbol:
symbol
VOO     11333.87
QQQM     1889.99
VXUS     1652.43
BND       376.16

Total invested to date, whole portfolio: $15,252.45


In [4]:
monthly = transactions.monthly_invested(trades)
monthly

symbol,BND,QQQM,VOO,VXUS,Total
month,,,,,
2026-01,0.00,0.00,96.06,0.00,96.06
2026-04,125.00,0.00,874.97,249.99,1249.96
2026-05,250.28,0.00,1749.96,499.99,2500.23
2026-06,0.88,1889.99,8612.88,902.45,11406.20


In [5]:
visualization.plot_monthly_invested(monthly).show()

In [6]:
daily = transactions.daily_investment_timeline(trades)
visualization.plot_daily_investment_timeline(daily).show()

In [7]:
pie_options = transactions.pie_chart_options(trades)
visualization.plot_investment_pie(pie_options).show()

## 3. Price history

One local cache file per symbol (`data/prices/{SYMBOL}.csv`), each call only fetching the date range missing since the last run — see `docs/architecture.md` for the cache design. History goes back to the first trade date across the whole portfolio.

In [8]:
symbols = sorted(trades["symbol"].unique())
first_trade_date = trades["trade_date"].min().date()

price_histories = prices.update_price_caches(
    symbols, since=first_trade_date, as_of=AS_OF_DATE, config=price_api_config
)
for symbol, history in price_histories.items():
    latest_close = history["close"].iloc[-1]
    print(f"{symbol}: {len(history)} trading days cached, latest close {latest_close:.2f}")

BND: 108 trading days cached, latest close 73.06
QQQM: 108 trading days cached, latest close 299.72
VOO: 108 trading days cached, latest close 687.08
VXUS: 108 trading days cached, latest close 84.71


## 4. Returns vs. a HYSA benchmark

For each trade: current price, days held, total return, CAGR-style annualized return, the compounded HYSA return over the same window (rate set by `returns_config.hysa_annual_rate`, default 4%), and the resulting alpha. See `docs/returns.md` for the derivation of each step.

In [9]:
def price_lookup(symbol: str, as_of: date) -> float | None:
    return prices.price_as_of(price_histories[symbol], as_of)


returns_df = returns.build_returns_table(
    trades, price_lookup, as_of=AS_OF_DATE, config=returns_config
)

display_table = returns_df[
    [
        "trade_date",
        "symbol",
        "usd_per_share",
        "current_price",
        "days_held",
        "total_return_pct",
        "annualized_return_pct",
        "hysa_period_return_pct",
        "alpha_period_pct",
    ]
].rename(columns={"usd_per_share": "price_paid"})
display_table

,trade_date,symbol,price_paid,current_price,days_held,total_return_pct,annualized_return_pct,hysa_period_return_pct,alpha_period_pct
0,2026-01-27,VOO,640.400000,687.080017,155,7.289197,18.019682,1.679485,5.609712
1,2026-04-10,BND,73.746313,73.059998,82,-0.930643,-4.076476,0.885016,-1.815659
2,2026-04-10,VOO,626.141405,687.080017,82,9.732404,51.195710,0.885016,8.847388
3,2026-04-10,VXUS,81.414056,84.709999,82,4.048371,19.321318,0.885016,3.163355
4,2026-05-05,BND,73.238072,73.059998,57,-0.243145,-1.546789,0.614367,-0.857512
5,2026-05-05,BND,73.250000,73.059998,57,-0.259389,-1.649404,0.614367,-0.873756
6,2026-05-05,VOO,665.865074,687.080017,57,3.186072,22.242640,0.614367,2.571705
7,2026-05-05,VXUS,83.529353,84.709999,57,1.413451,9.403943,0.614367,0.799083
8,2026-05-06,BND,74.358974,73.059998,56,-1.746900,-10.851502,0.603557,-2.350456
9,2026-06-01,QQQM,306.319287,299.720001,30,-2.154381,-23.277956,0.322882,-2.477264


In [10]:
numeric_cols = display_table.select_dtypes("number").columns
display_rounded = display_table.assign(**{c: display_table[c].round(2) for c in numeric_cols})
visualization.render_table(display_rounded, title=f"Returns as of {AS_OF_DATE}").show()

portfolio_alpha = returns.portfolio_alpha_pct(returns_df)
label = (
    f"Dollar-weighted portfolio alpha vs. {returns_config.hysa_annual_rate:.0%} HYSA "
    "(period, not annualized)"
)
print(f"{label}: {portfolio_alpha:+.2f}%")

Dollar-weighted portfolio alpha vs. 4% HYSA (period, not annualized): -0.20%


## 5. Annualized return curve

Per-trade annualized return against days held, with a fitted trend and the flat HYSA benchmark line. Short holds annualize into large, noisy numbers by design — that's why the combined alpha above uses period alpha instead of this annualized figure.

In [11]:
trend_x, trend_y = returns.fit_trend(
    returns_df["days_held"].to_numpy(),
    returns_df["annualized_return_pct"].to_numpy(),
    returns_config,
)
visualization.plot_return_curve(returns_df, trend_x, trend_y, returns_config).show()